In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd

# Load the dataset
file_path = "/kaggle/input/datasets/marcpaulo/titanic-huge-dataset-1m-passengers/huge_1M_titanic.csv"

df = pd.read_csv(file_path)

# Basic information
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nFirst 5 Rows:")
display(df.head())

## 1. Data Quality Report

The dataset was loaded and inspected to identify data quality issues before cleaning.

The following checks were performed:

- Dataset shape and number of records
- Column names
- Data types
- Missing values in each column
- Duplicate rows
- First few records for visual inspection

This initial inspection helps identify missing data, duplicate records, incorrect data types, and other potential data quality problems that need to be addressed during preprocessing.

In [ ]:
print("Numerical Summary:")
display(df.describe())

print("\nUnique Values:")
for column in df.columns:
    print(f"{column}: {df[column].nunique()} unique values")

## 2. Missing Value Handling

The dataset contains missing values in the `Age`, `Cabin`, and `Embarked` columns.

The following strategies were selected:

- **Age:** Missing values will be replaced with the median age because age is a numerical variable and the median is less affected by extreme values.
- **Cabin:** Missing values will be replaced with `"Unknown"` because a missing cabin represents unavailable information rather than a numerical value that can be estimated reliably.
- **Embarked:** Missing values will be replaced with the mode because it is a categorical variable and only a small number of records are missing.

These methods preserve the maximum number of records while providing reasonable replacements for missing information.

In [ ]:
# Store missing-value counts before cleaning
missing_before = df.isnull().sum()

# Handle missing Age using median
df["Age"] = df["Age"].fillna(df["Age"].median())

# Handle missing Cabin using "Unknown"
df["Cabin"] = df["Cabin"].fillna("Unknown")

# Handle missing Embarked using mode
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Check missing values after treatment
print("Missing values after handling:")
print(df.isnull().sum())

## 3. Duplicate Removal

Duplicate records were checked using all columns in the dataset.

The initial data quality report identified 0 duplicate rows. Therefore, no records were removed for duplication.

This check ensures that each transaction/record is retained only once.

In [ ]:
# Check for duplicate rows
duplicates_before = df.duplicated().sum()

print("Duplicate rows before removal:", duplicates_before)

# Remove duplicates if any exist
df = df.drop_duplicates().copy()

duplicates_after = df.duplicated().sum()

print("Duplicate rows after removal:", duplicates_after)
print("Dataset shape after duplicate handling:", df.shape)

## 4. Data Standardization

Categorical values were standardized by removing unnecessary whitespace and converting text values to a consistent format.

This helps ensure that equivalent categories are not treated as different values because of formatting inconsistencies.

In [ ]:
# Standardize categorical columns

df["Sex"] = df["Sex"].astype(str).str.strip().str.lower()
df["Embarked"] = df["Embarked"].astype(str).str.strip().str.upper()

print("Unique Sex values:")
print(df["Sex"].unique())

print("\nUnique Embarked values:")
print(df["Embarked"].unique())

In [ ]:
# Standardize column names

df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
)

print("Standardized column names:")
print(df.columns.tolist())

## 5. Outlier Detection and Handling

Outliers were identified in the numerical variables `Age` and `Fare` using the Interquartile Range (IQR) method.

The IQR method identifies values below Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR as potential outliers.

Instead of deleting these records, extreme values will be capped at the IQR boundaries. This preserves the records while reducing the influence of extreme observations on statistical analysis.

In [ ]:
# Function to detect outliers using IQR

def detect_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = data[
        (data[column] < lower_bound) |
        (data[column] > upper_bound)
    ]

    return Q1, Q3, lower_bound, upper_bound, len(outliers)


for column in ["Age", "Fare"]:
    Q1, Q3, lower, upper, count = detect_outliers(df, column)

    print(f"\n{column}")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("Lower Bound:", lower)
    print("Upper Bound:", upper)
    print("Number of outliers:", count)

In [ ]:
# Cap outliers using IQR boundaries

for column in ["Age", "Fare"]:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[column] = df[column].clip(
        lower=lower_bound,
        upper=upper_bound
    )

print("Outlier handling completed.")

print("\nAge range after capping:")
print(df["Age"].min(), "to", df["Age"].max())

print("\nFare range after capping:")
print(df["Fare"].min(), "to", df["Fare"].max())

## 6. Data Type Correction

Data types were reviewed to ensure that each column uses an appropriate representation. Identifier and categorical columns are converted to suitable types, while numerical variables remain numeric.

Correct data types improve memory usage, consistency, and reliability during analysis.

In [ ]:
# Correct data types

df["PassengerId"] = df["PassengerId"].astype("int64")
df["Survived"] = df["Survived"].astype("int64")
df["Pclass"] = df["Pclass"].astype("int64")
df["Age"] = df["Age"].astype("float64")
df["SibSp"] = df["SibSp"].astype("int64")
df["Parch"] = df["Parch"].astype("int64")
df["Fare"] = df["Fare"].astype("float64")

# Convert categorical columns
df["Sex"] = df["Sex"].astype("category")
df["Embarked"] = df["Embarked"].astype("category")

print("Data types after correction:")
print(df.dtypes)

In [ ]:
# Final data quality check

print("Final Dataset Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nFinal Data Types:")
print(df.dtypes)

In [ ]:
# Before vs After cleaning summary

summary = pd.DataFrame({
    "Metric": [
        "Number of Rows",
        "Number of Columns",
        "Missing Values",
        "Duplicate Rows"
    ],
    "Before Cleaning": [
        1000000,
        12,
        198600 + 770195 + 2240,
        0
    ],
    "After Cleaning": [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        df.duplicated().sum()
    ]
})

display(summary)

## 7. Before vs After Cleaning Summary

The data quality checks were repeated after cleaning to verify that the identified issues were successfully addressed.

Missing values were handled using appropriate statistical or categorical methods, duplicate records were checked and removed if present, categorical values were standardized, numerical outliers were capped using the IQR method, and data types were corrected.

The final dataset is ready for further analysis.

## 8. Export Cleaned Dataset

The cleaned dataset is exported as a CSV file so that it can be reused for further analysis and shared as the final cleaned output.

In [ ]:
# Save cleaned dataset as CSV

output_file = "/kaggle/working/cleaned_titanic_dataset.csv"

df.to_csv(output_file, index=False)

print("Cleaned dataset saved successfully!")
print("File:", output_file)

In [ ]:
# Verify exported file

cleaned_df = pd.read_csv(output_file)

print("Exported dataset shape:", cleaned_df.shape)
print("\nMissing values in exported dataset:")
print(cleaned_df.isnull().sum().sum())

display(cleaned_df.head())

In [ ]:
import matplotlib.pyplot as plt

## 9. Cleaning Results Visualization

The following visualizations compare the dataset before and after cleaning and demonstrate the effect of the preprocessing steps.

In [ ]:
# Missing values before vs after cleaning

missing_before_total = missing_before.sum()
missing_after_total = df.isnull().sum().sum()

plt.figure(figsize=(7, 5))

plt.bar(
    ["Before Cleaning", "After Cleaning"],
    [missing_before_total, missing_after_total]
)

plt.xlabel("Dataset State")
plt.ylabel("Number of Missing Values")
plt.title("Missing Values Before vs After Cleaning")
plt.show()

In [ ]:
# Age distribution after cleaning

plt.figure(figsize=(8, 5))

plt.hist(df["Age"], bins=30)

plt.xlabel("Age")
plt.ylabel("Number of Passengers")
plt.title("Age Distribution After Cleaning")
plt.show()

In [ ]:
# Fare distribution after cleaning

plt.figure(figsize=(8, 5))

plt.hist(df["Fare"], bins=30)

plt.xlabel("Fare")
plt.ylabel("Number of Passengers")
plt.title("Fare Distribution After Cleaning")
plt.show()

# Conclusion

The Titanic dataset was successfully cleaned and prepared for further analysis.

The data quality assessment identified significant missing values in the Age, Cabin, and Embarked columns. Age values were replaced using the median, missing Cabin values were labelled as "Unknown", and missing Embarked values were replaced using the mode.

Duplicate records were checked and no duplicates were found. Categorical values were standardized, column names were checked, and appropriate data types were applied.

Potential outliers in Age and Fare were identified using the IQR method and capped at the calculated boundaries instead of being removed. This preserved the dataset while reducing the influence of extreme values.

After cleaning, the dataset contains 1,000,000 rows and 12 columns with zero missing values and zero duplicate rows.

The cleaned dataset was exported as `cleaned_titanic_dataset.csv` and is ready for further analysis.